# HE_Hedge (Slave/Follower EA) — Component Testing

This notebook breaks down **HE_Hedge.mq5** (the Slave/Follower EA) into testable components.
Each section maps to a logical module in the MQ5 source so you can validate behavior
before deploying to a MetaTrader 5 terminal via the Hedge Edge App Connector.

**Source:** `agents/mt5/HedgEdge property/Developer tools/HE_Hedge.mq5` (v3.10)

## Architecture Overview
```
┌──────────────┐    ZMQ PUB/SUB    ┌──────────────┐
│  HE_Prop     │ ──────────────────>│  HE_Hedge    │
│  (Master)    │   tcp:51810        │  (Slave)     │
│  Publishes   │                    │  Subscribes  │
│  trade events│   ZMQ REQ/REP      │  & copies    │
│              │ <──────────────>   │  trades      │
└──────────────┘   tcp:51811        └──────┬───────┘
                                          │
                                    ZMQ REP :51821
                                          │
                                   ┌──────┴───────┐
                                   │  Electron    │
                                   │  App (UI)    │
                                   └──────────────┘
```

---
## Component 1 — Input Parameters & Configuration

The Slave EA accepts these input groups. Below we model them as a Python dict
so we can feed them into downstream component tests.

In [ ]:
# === Component 1: Input Parameters (mirrors MQ5 `input` declarations) ===

slave_config = {
    # --- License Settings ---
    "InpLicenseKey": "",                       # License Key (blank = read from shared file)
    "InpDeviceId": "",                          # Device ID (auto-generated if blank)
    "InpEndpointUrl": "https://hedge-edge-app-backend-production.up.railway.app/v1/license/validate",
    "InpPollIntervalSeconds": 300,              # License re-check every 5 min
    "InpDevMode": True,                         # << DEV MODE ON for testing

    # --- Master Connection ---
    "InpMasterAddress": "localhost",
    "InpMasterDataPort": 51810,                 # Master PUB port
    "InpMasterCommandPort": 51811,              # Master REP port
    "InpEnableCurve": False,
    "InpMasterPublicKey": "",

    # --- Trade Copy Settings ---
    "InpLotMultiplier": 1.0,
    "InpFixedLots": 0.0,                        # 0 = use multiplier
    "InpMaxLots": 100.0,
    "InpSlippage": 10,
    "InpMagicNumber": 123456,
    "InpTradeComment": "HE-Copy",
    "InpCopySLTP": True,
    "InpInvertTrades": True,                    # ALWAYS true for hedge copier
    "InpCopyCloseSignals": True,

    # --- App Communication ---
    "InpCommandPort": 51821,                    # Local REP for Electron app
    "InpEnableLocalCommands": True,
}

print("Slave config loaded:")
for k, v in slave_config.items():
    print(f"  {k}: {v}")

---
## Component 2 — Shared License Key (Common Files)

The EA reads a shared license key from `%APPDATA%\MetaQuotes\Terminal\Common\Files\HedgeEdge\license.key`.
This allows the Electron app to write the key once and all EAs auto-read it.

**MQ5 Function:** `ReadSharedLicenseKey()`

In [ ]:
import os

def find_mt5_common_files():
    """Locate MT5 Common Files directory."""
    appdata = os.environ.get("APPDATA", "")
    common_path = os.path.join(appdata, "MetaQuotes", "Terminal", "Common", "Files")
    if os.path.isdir(common_path):
        return common_path
    return None

def read_shared_license_key():
    """Mirror of MQ5 ReadSharedLicenseKey() — reads license.key from Common Files."""
    common = find_mt5_common_files()
    if not common:
        print("MT5 Common Files directory not found")
        return ""
    
    key_file = os.path.join(common, "HedgeEdge", "license.key")
    if not os.path.isfile(key_file):
        print(f"No shared license file at: {key_file}")
        return ""
    
    with open(key_file, "r", encoding="utf-8") as f:
        key = f.read().strip()
    
    if len(key) < 8:
        print("Shared license key too short")
        return ""
    
    masked = key[:4] + "****" + key[-4:]
    print(f"Shared license key loaded ({len(key)} chars): {masked}")
    return key

# --- Test ---
common_dir = find_mt5_common_files()
print(f"MT5 Common Files: {common_dir}")
shared_key = read_shared_license_key()

---
## Component 3 — License Validation (WebRequest Fallback)

When `HedgeEdgeLicense.dll` isn't available, the EA falls back to HTTP `WebRequest`
against the Railway backend. We test this endpoint directly.

**MQ5 Function:** `ValidateLicenseViaWebRequest()`

In [ ]:
import requests

def validate_license_via_web(license_key: str, account_id: str = "12345",
                              broker: str = "TestBroker", device_id: str = "dev-notebook"):
    """Mirror of MQ5 ValidateLicenseViaWebRequest()."""
    url = slave_config["InpEndpointUrl"]
    payload = {
        "licenseKey": license_key,
        "accountId": account_id,
        "broker": broker,
        "deviceId": device_id,
    }
    
    masked = license_key[:4] + "****" + license_key[-4:] if len(license_key) > 8 else "***"
    print(f"Validating key={masked} at {url}")
    
    try:
        resp = requests.post(url, json=payload, timeout=10)
        print(f"HTTP {resp.status_code}")
        data = resp.json()
        print(f"Response: {data}")
        return data.get("valid", False)
    except Exception as e:
        print(f"Request failed: {e}")
        return False

# --- Test (replace with a real key or use DEV MODE) ---
if slave_config["InpDevMode"]:
    print("DEV MODE: License validation skipped")
else:
    key = shared_key or slave_config["InpLicenseKey"]
    if key:
        validate_license_via_web(key)
    else:
        print("No license key available — set InpLicenseKey or place in Common Files")

---
## Component 4 — ZMQ Subscriber Connection

The Slave subscribes to the Master's PUB socket on `tcp://<master>:51810`
and filters on topics `EVENT|` and `SNAPSHOT|`.

**MQ5 Function:** `InitializeZMQ()` (subscriber side)

> **Prereq:** Install `pyzmq` — `pip install pyzmq`

In [ ]:
import zmq
import json as json_mod

def create_subscriber(master_address: str, master_port: int):
    """Mirror of MQ5 InitializeZMQ() — SUB socket creation."""
    ctx = zmq.Context()
    sub = ctx.socket(zmq.SUB)
    endpoint = f"tcp://{master_address}:{master_port}"
    
    sub.setsockopt(zmq.LINGER, 100)
    sub.setsockopt(zmq.RCVHWM, 10000)
    sub.setsockopt(zmq.RCVTIMEO, 1)  # 1ms timeout for non-blocking
    
    # Subscribe to both topics
    sub.setsockopt_string(zmq.SUBSCRIBE, "EVENT|")
    sub.setsockopt_string(zmq.SUBSCRIBE, "SNAPSHOT|")
    
    sub.connect(endpoint)
    print(f"SUB socket connected to {endpoint}")
    print(f"  Subscribed to: EVENT|, SNAPSHOT|")
    return ctx, sub

# --- Test: Create subscriber (will wait for a master) ---
addr = slave_config["InpMasterAddress"]
port = slave_config["InpMasterDataPort"]
print(f"Creating subscriber to {addr}:{port}...")
# Uncomment to actually connect:
# zmq_ctx, zmq_sub = create_subscriber(addr, port)

---
## Component 5 — Event Processing & Routing

Incoming messages are split by topic, then the JSON payload is routed
by `type` field to the appropriate handler.

**MQ5 Functions:** `ProcessMasterEvents()`, `HandleEvent()`

Supported event types:
- `POSITION_OPENED` — copy a new trade (inverted)
- `POSITION_CLOSED` — close the mapped slave position
- `POSITION_MODIFIED` — update SL/TP on slave
- `POSITION_REVERSED` — close + reopen
- `HEARTBEAT` — keepalive
- `CONNECTED` / `DISCONNECTED` — master lifecycle
- `ACCOUNT_UPDATE` — periodic reconciliation

In [ ]:
# === Component 5: Event Router ===

def extract_json_value(json_str: str, key: str) -> str:
    """Simple JSON value extractor (mirrors MQ5 ExtractJsonValue)."""
    import json as j
    try:
        data = j.loads(json_str)
        val = data.get(key, "")
        return str(val) if val is not None else ""
    except:
        return ""

def route_event(topic: str, message: str):
    """Mirror of ProcessMasterEvents + HandleEvent routing."""
    if topic == "EVENT":
        data = json_mod.loads(message)
        event_type = data.get("type", "")
        handlers = {
            "POSITION_OPENED": handle_position_opened,
            "POSITION_CLOSED": handle_position_closed,
            "POSITION_MODIFIED": handle_position_modified,
            "POSITION_REVERSED": handle_position_reversed,
            "HEARTBEAT": handle_heartbeat,
            "CONNECTED": handle_master_connected,
            "DISCONNECTED": handle_master_disconnected,
            "ACCOUNT_UPDATE": handle_account_update,
        }
        handler = handlers.get(event_type)
        if handler:
            print(f"  -> Routing to {handler.__name__}")
            handler(data)
        else:
            print(f"  -> Unhandled event type: {event_type}")
    elif topic == "SNAPSHOT":
        print(f"  -> Routing to handle_snapshot")
        handle_snapshot(json_mod.loads(message))
    else:
        print(f"  -> Unknown topic: {topic}")

# Stub handlers (implemented in later components)
def handle_position_opened(data): print(f"    POSITION_OPENED: {data.get('data', {}).get('symbol', '?')}")
def handle_position_closed(data): print(f"    POSITION_CLOSED")
def handle_position_modified(data): print(f"    POSITION_MODIFIED")
def handle_position_reversed(data): print(f"    POSITION_REVERSED")
def handle_heartbeat(data): print(f"    HEARTBEAT")
def handle_master_connected(data): print(f"    MASTER CONNECTED")
def handle_master_disconnected(data): print(f"    MASTER DISCONNECTED")
def handle_account_update(data): print(f"    ACCOUNT_UPDATE")
def handle_snapshot(data): print(f"    SNAPSHOT")

# --- Test: Simulate events ---
test_events = [
    ("EVENT", json_mod.dumps({"type": "HEARTBEAT", "data": {"balance": 10000}})),
    ("EVENT", json_mod.dumps({"type": "POSITION_OPENED", "data": {"symbol": "EURUSD", "type": "BUY", "volume": 0.10, "position": 12345}})),
    ("EVENT", json_mod.dumps({"type": "POSITION_CLOSED", "data": {"position": 12345, "profit": 42.50}})),
    ("SNAPSHOT", json_mod.dumps({"type": "SNAPSHOT", "positions": []})),
]

print("=== Event Routing Test ===")
for topic, msg in test_events:
    print(f"\n[{topic}] received:")
    route_event(topic, msg)

---
## Component 6 — Trade Inversion Logic (Core Hedge)

The defining feature: when `InpInvertTrades=true` (default), the slave
**inverts** the direction and **swaps SL/TP** so the follower profits when the leader loses.

```
Master BUY  → Slave SELL  (SL ↔ TP swapped)
Master SELL → Slave BUY   (SL ↔ TP swapped)
```

**MQ5 location:** Inside `HandlePositionOpened()`

In [ ]:
# === Component 6: Trade Inversion Logic ===

def invert_trade(side: str, sl: float, tp: float, invert: bool = True):
    """Mirror of the inversion block inside HandlePositionOpened()."""
    if invert:
        side = "SELL" if side == "BUY" else "BUY"
        sl, tp = tp, sl  # Leader's SL -> Follower's TP and vice versa
    return side, sl, tp

# --- Test ---
test_cases = [
    {"side": "BUY",  "sl": 1.0800, "tp": 1.1000},
    {"side": "SELL", "sl": 1.1000, "tp": 1.0800},
    {"side": "BUY",  "sl": 0.0,    "tp": 1.1000},  # No SL
]

print("=== Inversion Logic Test ===")
print(f"{'Master':>10} {'SL':>10} {'TP':>10}  =>  {'Slave':>10} {'SL':>10} {'TP':>10}")
print("-" * 70)
for tc in test_cases:
    new_side, new_sl, new_tp = invert_trade(tc["side"], tc["sl"], tc["tp"])
    print(f"{tc['side']:>10} {tc['sl']:>10.4f} {tc['tp']:>10.4f}  =>  {new_side:>10} {new_sl:>10.4f} {new_tp:>10.4f}")

---
## Component 7 — Lot Size Calculation

Lot sizing follows this priority:
1. If `fixedLots > 0`, use that
2. Otherwise `masterVolume × lotMultiplier`
3. Clamped to `[lotMin, lotMax]` and rounded to `lotStep`

**MQ5 Function:** `CalculateLotSize()`

In [ ]:
# === Component 7: Lot Size Calculation ===
import math

def calculate_lot_size(master_volume: float,
                       lot_multiplier: float = 1.0,
                       fixed_lots: float = 0.0,
                       max_lots: float = 100.0,
                       lot_step: float = 0.01,
                       lot_min: float = 0.01,
                       lot_max: float = 500.0) -> float:
    """Mirror of MQ5 CalculateLotSize()."""
    if fixed_lots > 0:
        lots = fixed_lots
    else:
        lots = master_volume * lot_multiplier
    
    if lot_step > 0:
        lots = math.floor(lots / lot_step) * lot_step
    
    lots = max(lot_min, lots)
    lots = min(lot_max, lots)
    lots = min(max_lots, lots)
    
    return round(lots, 2)

# --- Test ---
test_scenarios = [
    {"master_volume": 0.10, "lot_multiplier": 1.0, "fixed_lots": 0.0, "desc": "1:1 copy"},
    {"master_volume": 0.10, "lot_multiplier": 2.0, "fixed_lots": 0.0, "desc": "2x multiplier"},
    {"master_volume": 0.10, "lot_multiplier": 0.5, "fixed_lots": 0.0, "desc": "0.5x multiplier"},
    {"master_volume": 5.00, "lot_multiplier": 1.0, "fixed_lots": 0.01, "desc": "Fixed 0.01"},
    {"master_volume": 0.001, "lot_multiplier": 1.0, "fixed_lots": 0.0, "desc": "Below min"},
]

print("=== Lot Size Calculation Test ===")
for s in test_scenarios:
    result = calculate_lot_size(s["master_volume"], s["lot_multiplier"], s["fixed_lots"])
    print(f"  {s['desc']:20s} master={s['master_volume']:.2f} mult={s['lot_multiplier']:.1f} fixed={s['fixed_lots']:.2f} => {result:.2f} lots")

---
## Component 8 — Position Mapping (Master → Slave)

The slave maintains a mapping of `masterTicket → slaveTicket` to know which
local position corresponds to which master position. This is critical for
close/modify signals.

**MQ5 Struct:** `PositionMap g_positionMap[]`

In [ ]:
# === Component 8: Position Mapping ===
from dataclasses import dataclass, field

@dataclass
class PositionMap:
    master_ticket: int
    slave_ticket: int
    symbol: str
    volume: float
    side: str  # BUY or SELL (after inversion)

class PositionTracker:
    """Mirrors g_positionMap[] with add/remove/lookup."""
    def __init__(self):
        self.mappings: list[PositionMap] = []
    
    def add(self, master_ticket, slave_ticket, symbol, volume, side):
        # Duplicate check (mirrors HandlePositionOpened guard)
        for m in self.mappings:
            if m.master_ticket == master_ticket:
                print(f"  Duplicate POSITION_OPENED for master #{master_ticket} — ignoring")
                return False
        self.mappings.append(PositionMap(master_ticket, slave_ticket, symbol, volume, side))
        return True
    
    def find_by_master(self, master_ticket) -> PositionMap | None:
        for m in self.mappings:
            if m.master_ticket == master_ticket:
                return m
        return None
    
    def remove_by_index(self, index):
        if 0 <= index < len(self.mappings):
            removed = self.mappings.pop(index)
            print(f"  Removed mapping: master #{removed.master_ticket} -> slave #{removed.slave_ticket}")
    
    def remove_by_master(self, master_ticket):
        for i, m in enumerate(self.mappings):
            if m.master_ticket == master_ticket:
                self.remove_by_index(i)
                return True
        return False
    
    def __repr__(self):
        lines = [f"PositionTracker ({len(self.mappings)} mapped):"]
        for m in self.mappings:
            lines.append(f"  master #{m.master_ticket} -> slave #{m.slave_ticket} {m.symbol} {m.side} {m.volume}")
        return "\n".join(lines)

# --- Test ---
tracker = PositionTracker()
tracker.add(100001, 200001, "EURUSD", 0.10, "SELL")
tracker.add(100002, 200002, "GBPUSD", 0.05, "BUY")
tracker.add(100001, 200099, "EURUSD", 0.10, "SELL")  # duplicate
print(tracker)
print(f"\nLookup master #100002: {tracker.find_by_master(100002)}")
tracker.remove_by_master(100001)
print(tracker)

---
## Component 9 — Trade Log (JSONL to Common Files)

Every copied trade is written to `HedgeEdge/trade_log_<login>.jsonl` in Common Files.
The Electron app reads these on startup to reconcile missed trades.

**MQ5 Function:** `WriteTradeLogEntry()`

In [ ]:
import json as json_mod
from datetime import datetime

def build_trade_log_entry(event_type: str, symbol: str, side: str, volume: float,
                          profit: float, swap: float, commission: float,
                          master_ticket: int, slave_ticket: int,
                          entry_price: float, close_price: float,
                          account: str = "12345", broker: str = "TestBroker",
                          balance: float = 10000.0, equity: float = 10000.0) -> str:
    """Mirror of MQ5 WriteTradeLogEntry() — builds the JSONL line."""
    entry = {
        "event": event_type,
        "account": account,
        "broker": broker,
        "masterTicket": master_ticket,
        "slaveTicket": slave_ticket,
        "symbol": symbol,
        "side": side,
        "volume": round(volume, 2),
        "entryPrice": round(entry_price, 5),
        "closePrice": round(close_price, 5),
        "profit": round(profit, 2),
        "swap": round(swap, 2),
        "commission": round(commission, 2),
        "balance": round(balance, 2),
        "equity": round(equity, 2),
        "timestamp": datetime.now().strftime("%Y.%m.%d %H:%M:%S"),
        "timestampUnix": int(datetime.now().timestamp()),
    }
    return json_mod.dumps(entry)

# --- Test ---
line1 = build_trade_log_entry("COPY_OPEN", "EURUSD", "SELL", 0.10, 0, 0, 0, 100001, 200001, 1.09500, 0)
line2 = build_trade_log_entry("COPY_CLOSE", "EURUSD", "SELL", 0.10, 42.50, -1.20, -0.70, 100001, 200001, 1.09500, 1.09100)

print("=== Trade Log Entries (JSONL) ===")
print(line1)
print(line2)

# Verify they parse
for line in [line1, line2]:
    parsed = json_mod.loads(line)
    print(f"  ✓ {parsed['event']} {parsed['symbol']} {parsed['side']} vol={parsed['volume']} P&L={parsed['profit']}")

---
## Component 10 — App Command Channel (REP Socket)

The Electron app communicates with the slave via a ZMQ REQ/REP socket on port 51821.
Supported commands:

| Action | Description |
|--------|-------------|
| `PAUSE` | Pause trade copying |
| `RESUME` | Resume trade copying |
| `STATUS` | Get full slave status |
| `PING` | Keepalive check |
| `CONFIG` | Get current config |
| `SET_CONFIG` | Push runtime config (invertTrades, lotMultiplier, etc.) |
| `OPEN_POSITION` | Direct trade from app |
| `MODIFY_POSITION` | Modify SL/TP |
| `CLOSE_POSITION` | Close by ticket |
| `CLOSE_ALL` | Close all positions |

**MQ5 Function:** `ProcessLocalCommands()`

In [ ]:
# === Component 10: App Command Channel Test ===

def send_command(port: int, command: dict, timeout_ms: int = 3000) -> dict | None:
    """Send a JSON command to the slave's REP socket and get the response."""
    ctx = zmq.Context()
    req = ctx.socket(zmq.REQ)
    req.setsockopt(zmq.LINGER, 100)
    req.setsockopt(zmq.RCVTIMEO, timeout_ms)
    req.setsockopt(zmq.SNDTIMEO, timeout_ms)
    
    endpoint = f"tcp://localhost:{port}"
    req.connect(endpoint)
    
    try:
        req.send_string(json_mod.dumps(command))
        reply = req.recv_string()
        return json_mod.loads(reply)
    except zmq.error.Again:
        print(f"Timeout waiting for reply from {endpoint}")
        return None
    finally:
        req.close()
        ctx.term()

# --- Test commands (run when MT5 EA is active) ---
cmd_port = slave_config["InpCommandPort"]
print(f"Command port: {cmd_port}")
print("\nTest commands (uncomment to run against live EA):")
print("  send_command(cmd_port, {'action': 'PING'})")
print("  send_command(cmd_port, {'action': 'STATUS'})")
print("  send_command(cmd_port, {'action': 'CONFIG'})")
print("  send_command(cmd_port, {'action': 'SET_CONFIG', 'invertTrades': 'true', 'lotMultiplier': '1.5'})")
print("  send_command(cmd_port, {'action': 'PAUSE'})")
print("  send_command(cmd_port, {'action': 'RESUME'})")

# Uncomment to test against running EA:
# result = send_command(cmd_port, {"action": "PING"})
# print(f"PING result: {result}")

---
## Component 11 — Position Reconciliation

On snapshot/account updates, the slave reconciles its positions against the master's.
Missing positions are opened, orphaned positions are closed.

**MQ5 Function:** `ReconcilePositions()`

In [ ]:
# === Component 11: Reconciliation Logic ===

def reconcile(master_positions: list[dict], tracker: PositionTracker, invert: bool = True):
    """Simulate reconciliation — find missing and orphaned positions."""
    to_open = []   # master positions we don't have mapped
    to_close = []  # slave positions whose master no longer exists
    
    # 1. Find master positions we haven't copied yet
    master_tickets = set()
    for mp in master_positions:
        mt = mp["id"]
        master_tickets.add(mt)
        if not tracker.find_by_master(mt):
            side = mp["side"]
            sl = mp.get("stopLoss", 0) or 0
            tp = mp.get("takeProfit", 0) or 0
            if invert:
                side = "SELL" if side == "BUY" else "BUY"
                sl, tp = tp, sl
            to_open.append({"master_ticket": mt, "symbol": mp["symbol"],
                            "side": side, "volume": mp.get("volumeLots", 0.01),
                            "sl": sl, "tp": tp})
    
    # 2. Find orphaned slave positions
    for m in tracker.mappings:
        if m.master_ticket not in master_tickets:
            to_close.append(m)
    
    return to_open, to_close

# --- Test ---
tracker2 = PositionTracker()
tracker2.add(1001, 2001, "EURUSD", 0.10, "SELL")
tracker2.add(1002, 2002, "GBPUSD", 0.05, "BUY")

master_state = [
    {"id": 1001, "symbol": "EURUSD", "side": "BUY", "volumeLots": 0.10},   # already mapped
    {"id": 1003, "symbol": "USDJPY", "side": "SELL", "volumeLots": 0.20},  # NEW - needs copy
]
# Note: master 1002 is gone → slave 2002 should be closed

opens, closes = reconcile(master_state, tracker2)
print("=== Reconciliation Result ===")
print(f"Positions to OPEN ({len(opens)}):")
for o in opens:
    print(f"  {o['symbol']} {o['side']} {o['volume']} lots (master #{o['master_ticket']})")
print(f"Positions to CLOSE ({len(closes)}):")
for c in closes:
    print(f"  slave #{c.slave_ticket} {c.symbol} (master #{c.master_ticket} gone)")

---
## Component 12 — Registration File (Electron App Discovery)

The EA writes a JSON file to Common Files: `HedgeEdge/<login>.json`
so the Electron app can auto-discover connected terminals.

**MQ5 Function:** `WriteRegistrationFile()`

In [ ]:
# === Component 12: Registration File ===

def build_slave_registration(login: str, broker: str, server: str, config: dict) -> dict:
    """Mirror of MQ5 WriteRegistrationFile() for the slave."""
    return {
        "login": login,
        "broker": broker,
        "server": server,
        "commandPort": config["InpCommandPort"],
        "role": "slave",
        "version": "3.0",
        "eventDriven": True,
        "masterAddress": config["InpMasterAddress"],
        "masterDataPort": config["InpMasterDataPort"],
        "curveEnabled": config["InpEnableCurve"],
        "timestamp": datetime.now().strftime("%Y.%m.%d %H:%M:%S"),
    }

def check_registration_files():
    """Check for existing registration files in Common Files."""
    common = find_mt5_common_files()
    if not common:
        print("Common Files not found")
        return []
    
    he_dir = os.path.join(common, "HedgeEdge")
    if not os.path.isdir(he_dir):
        print(f"No HedgeEdge directory in Common Files")
        return []
    
    files = [f for f in os.listdir(he_dir) if f.endswith(".json")]
    print(f"Found {len(files)} registration file(s):")
    for f in files:
        path = os.path.join(he_dir, f)
        with open(path, "r") as fh:
            data = json_mod.load(fh)
        print(f"  {f}: role={data.get('role')} login={data.get('login')} port={data.get('commandPort', data.get('dataPort'))}")
    return files

# --- Test ---
reg = build_slave_registration("12345", "TestBroker", "TestServer", slave_config)
print("Registration JSON:")
print(json_mod.dumps(reg, indent=2))

print("\n--- Checking existing registrations ---")
check_registration_files()

---
## Component 13 — End-to-End Integration Test

Brings all components together: connect to master, receive events, invert, copy.
Run this cell with both the HE_Prop (master) and HE_Hedge (slave) EAs running on MT5.

In [ ]:
# === Component 13: End-to-End Integration ===
import time

def integration_test(master_addr: str, master_port: int, slave_cmd_port: int,
                     listen_seconds: int = 10):
    """Listen for master events and test command channel."""
    print(f"=== Integration Test ===")
    print(f"Master: tcp://{master_addr}:{master_port}")
    print(f"Slave cmd: tcp://localhost:{slave_cmd_port}")
    
    # 1. Test command channel (PING)
    print("\n[1] Testing PING to slave EA...")
    ping_result = send_command(slave_cmd_port, {"action": "PING"})
    if ping_result and ping_result.get("pong"):
        print(f"  Slave responded: role={ping_result.get('role')}")
    else:
        print(f"  Slave not responding (is HE_Hedge.mq5 running on MT5?)")
        return
    
    # 2. Get STATUS
    print("\n[2] Getting slave STATUS...")
    status = send_command(slave_cmd_port, {"action": "STATUS"})
    if status:
        print(f"  Account: {status.get('accountId')}")
        print(f"  Balance: {status.get('balance')}")
        print(f"  Master Connected: {status.get('masterConnected')}")
        print(f"  Trades Copied: {status.get('tradesCopied')}")
        print(f"  Positions: {len(status.get('positions', []))}")
    
    # 3. Listen for master events
    print(f"\n[3] Subscribing to master for {listen_seconds}s...")
    ctx, sub = create_subscriber(master_addr, master_port)
    
    events = []
    end_time = time.time() + listen_seconds
    while time.time() < end_time:
        try:
            raw = sub.recv_string(flags=zmq.NOBLOCK)
            topic, payload = raw.split("|", 1)
            data = json_mod.loads(payload)
            events.append((topic, data.get("type", "?")))
            print(f"  [{topic}] {data.get('type', '?')}")
        except zmq.error.Again:
            time.sleep(0.05)
    
    sub.close()
    ctx.term()
    print(f"\n  Received {len(events)} events in {listen_seconds}s")

# --- Run (uncomment when EAs are running) ---
# integration_test(
#     master_addr=slave_config["InpMasterAddress"],
#     master_port=slave_config["InpMasterDataPort"],
#     slave_cmd_port=slave_config["InpCommandPort"],
#     listen_seconds=15
# )

---
## Dashboard (Component 14) — Visual Panel

The MQ5 renders a branded overlay on the MT5 chart. The key status fields are:
- **Status**: Connected / Paused / License Error / Waiting
- **Mode**: Reverse (inverted) or Mirror
- **Lots**: Fixed or multiplier
- **Stats**: Copied / Failed counts

This is purely MQL5 `OBJ_RECTANGLE_LABEL` / `OBJ_LABEL` rendering — no Python equivalent needed.
The registration file + command channel give the Electron app everything it needs.

---
## Deployment Checklist

Before deploying HE_Hedge.mq5 to a live MT5 terminal:

1. **DLL files in place:**
   - `MQL5/Libraries/libzmq.dll`
   - `MQL5/Libraries/libsodium.dll`
   - `MQL5/Libraries/HedgeEdgeLicense.dll` (optional)

2. **Include files:**
   - `MQL5/Include/ZMQv2.mqh`

3. **MT5 Settings:**
   - Tools > Options > Expert Advisors > Allow DLL imports
   - Tools > Options > Expert Advisors > Allow WebRequest for: `https://hedge-edge-app-backend-production.up.railway.app`

4. **License:**
   - Place key in EA input OR in `Common Files/HedgeEdge/license.key`
   - OR enable DEV MODE for testing

5. **Compile:** Use MQL Tools extension in VS Code or MT5 MetaEditor

6. **Attach:** Drag compiled `.ex5` onto a chart with the EA inputs configured